# 2. Copernicus bağlantısını kontrol et

Colab'in sol menüsündeki **anahtar simgesini (Secrets)** açın. Önceki projede kullandığınız iki değeri, aşağıdaki adlarla ekleyin ve bu defter için **Notebook erişimi** seçeneğini açın:

- `CDSE_CLIENT_ID`
- `CDSE_CLIENT_SECRET`

Değerleri bilgisayarınızdaki projenin `.env` dosyasından kendiniz kopyalayabilirsiniz. Bunlar Copernicus hesap şifreniz değil, Sentinel Hub OAuth istemcisinin iki anahtarıdır. `.env` dosyasını bu deftere veya Drive'a yüklemeniz gerekmez.

Anahtarlar elinizde yoksa [Copernicus panelindeki](https://shapps.dataspace.copernicus.eu/dashboard/) **User Settings → OAuth clients** bölümünden bir istemci oluşturabilirsiniz. [Resmî kimlik doğrulama açıklaması](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Overview/Authentication.html).

Ardından **Çalışma zamanı → Tümünü çalıştır** seçeneğini kullanın. Bu defter yalnızca kimlik doğrulamayı kontrol eder. Görüntü indirmez, Drive bağlantısı kurmaz ve dosya oluşturmaz. İndirme kotası ve Process API erişimi daha sonraki küçük denemede ayrıca kontrol edilecektir.

In [ ]:
import os
from google.colab import userdata

CDSE_SECRET_NAMES = ('CDSE_CLIENT_ID', 'CDSE_CLIENT_SECRET')

def load_cdse_credentials():
    credentials = {}
    for name in CDSE_SECRET_NAMES:
        try:
            value = userdata.get(name)
        except userdata.SecretNotFoundError:
            raise RuntimeError(f'{name} bulunamadı. Colab Secrets bölümüne ekleyin.') from None
        except userdata.NotebookAccessError:
            raise RuntimeError(f'{name} için bu defterin Notebook erişimini açın.') from None
        except userdata.TimeoutException:
            raise RuntimeError('Secrets okunamadı. Hücreyi Colab ekranından yeniden çalıştırın.') from None
        if not isinstance(value, str) or not value.strip():
            raise RuntimeError(f'{name} boş. Secrets bölümündeki değeri kontrol edin.')
        credentials[name] = value.strip()
    os.environ.update(credentials)

load_cdse_credentials()
print('Copernicus anahtarları Colab belleğine yüklendi.')

In [ ]:
import requests

TOKEN_URL = (
    'https://identity.dataspace.copernicus.eu/auth/realms/CDSE'
    '/protocol/openid-connect/token'
)

def check_cdse_login():
    if any(not os.environ.get(name) for name in CDSE_SECRET_NAMES):
        raise RuntimeError('Önce anahtarları yükleyen hücreyi başarıyla çalıştırın.')
    try:
        response = requests.post(
            TOKEN_URL,
            data={
                'grant_type': 'client_credentials',
                'client_id': os.environ['CDSE_CLIENT_ID'],
                'client_secret': os.environ['CDSE_CLIENT_SECRET'],
            },
            timeout=(15, 45),
            allow_redirects=False,
        )
    except requests.RequestException:
        raise RuntimeError('Copernicus sunucusuna ulaşılamadı. Bir süre sonra yeniden deneyin.') from None
    try:
        if response.status_code in (400, 401, 403):
            raise RuntimeError(
                f'Kimlik doğrulama reddedildi (HTTP {response.status_code}). '
                'Anahtarların doğru ve OAuth istemcisinin geçerli olduğunu kontrol edin.'
            )
        if response.status_code == 429:
            raise RuntimeError('Copernicus istek sınırına ulaşıldı. Bekleyip yeniden deneyin.')
        if response.status_code != 200:
            raise RuntimeError(f'Copernicus bağlantısı tamamlanamadı (HTTP {response.status_code}).')
        try:
            payload = response.json()
        except ValueError:
            raise RuntimeError('Copernicus yanıtı beklenen biçimde değil. Daha sonra yeniden deneyin.') from None
        if not isinstance(payload, dict) or not payload.get('access_token'):
            raise RuntimeError('Copernicus erişim tokenı alınamadı.')
    finally:
        response.close()
    print('CDSE kimlik doğrulaması başarılı.')

check_cdse_login()

## Tamamlandığında

**CDSE kimlik doğrulaması başarılı.** yazısını görürseniz bağlantı adımı tamamdır. Sonraki aşama, küçük bir deneme için pipeline kurulumudur.

Anahtarları veya tokenı sohbetle paylaşmayın. Hata olursa yalnızca görünen kısa hata mesajını paylaşmanız yeterli.

# 3. Pipeline kurulumu

Kimlik doğrulama başarılı olduktan sonra bu hücreyi çalıştırın. Temel paket, PyPI'de yayımlanmış **0.3.2** sürümüne sabitlenir. Bu adım programı kurar ve temel kütüphaneleri kontrol eder; görüntü indirmez. GPU gerekmez.

Daha önce yüklediğiniz defter bu bölümü içermiyorsa aşağıdaki kodu aynı defterin sonuna **+ Kod** ile ekleyebilirsiniz.

In [ ]:
%pip -q install sentinel-crop-pipeline==0.3.2

import sentinel_crop_pipeline
import rasterio
import numpy

print(f'Pipeline kurulumu hazır. Sürüm: {sentinel_crop_pipeline.__version__}')

# 4. Bu çalışma zamanını Drive'a bağla

Aşağıdaki hücreyi çalıştırın. Hesap sorulursa, ilk adımda **TUBITAK_Urla** klasörünü oluşturduğunuz 5 TB alanı olan Google hesabını seçin.

Defterler farklı Colab çalışma zamanlarında açılabileceği için bu defterin de Drive'a bağlanması gerekir. Bu hücre mevcut proje klasörünü kontrol eder; görüntü indirmez.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/TUBITAK_Urla')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError('TUBITAK_Urla klasörü bulunamadı. Doğru Google hesabını seçtiğinizi kontrol edin.')
print('Drive bağlantısı hazır.')

# 5. Görüntü indirmeden katalog araması

Haziran 2025 ve mevcut geçici Urla dikdörtgeni yalnızca teknik deneme içindir. Bu dikdörtgen resmî ilçe sınırı değildir; araştırmanın asıl bölgesi ve veri yılı etiket kaynaklarıyla birlikte belirlenecektir.

[Copernicus STAC kataloğunda](https://documentation.dataspace.copernicus.eu/APIs/STAC.html) Sentinel-2 L2A adaylarını arıyoruz. Bu hücre görüntü indirmez, SCL/AI kontrolü yapmaz ve dosya oluşturmaz.

Bulut oranı tüm uydu sahnesinin metadata değeridir. Çalışma alanımızdaki bulut/gölge durumu henüz doğrulanmış değildir. Sonuç sayısı değişebilir.

In [ ]:
from shapely.geometry import box
from sentinel_crop_pipeline.discovery.providers.cdse import CdseProvider

pilot_aoi = box(26.60, 38.19, 26.95, 38.43)
pilot_provider = CdseProvider()
pilot_scenes = sorted(
    pilot_provider.search(
        pilot_aoi, '2025-06-01', '2025-06-30', max_cloud_cover=20
    ),
    key=lambda scene: (scene.datetime, scene.id),
)

print('Bulunan aday görüntü sayısı:', len(pilot_scenes))
for scene in pilot_scenes[:3]:
    print(scene.datetime[:10], 'Bulut oranı (%):', scene.cloud_cover)

# 6. Tek tarih için küçük görüntü denemesi

10 Haziran 2025 için küçük bir pencere indiriyoruz. Bu yalnızca teknik denemedir; bölgenin tarımsal sınıfları veya etiket yılı belirlenmiş değildir. İndirme Colab'in geçici diskine yapılır; dosya doğrulandıktan sonra ayrı bir adımda Drive'a aktarılacaktır.

B02/B03/B04/B08 bantlarına SCL kalite katmanı eklenir. SCL doğal olarak 20 m olduğundan, 10 m çıktı gridine **nearest** ile alınır. Bu adım normalizasyon veya indeks hesabı yapmaz.

[Process API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html) isteği aday ürünün tam kimliğini değil, UTC gününü filtreler. Bu nedenle çıktı belirli bir günün AOI görüntüsüdür; tek bir STAC ürününün birebir kopyası sayılmaz. Sağlayıcı `harmonizeValues` göndermediğinden, belgelenen sunucu varsayılanı olan `true` geçerlidir: DN değerleri harmonize edilir, negatif değerler sıfıra kırpılır. Ek bir offset düzeltmesi uygulamayın.

Dosya yapısının doğrulanması model eğitimi için kalite kabulü değildir; bulut/gölge oranı ve görsel inceleme ayrıca değerlendirilecektir.

In [ ]:
from tempfile import mkdtemp
from shapely.geometry import box, shape
import numpy as np
import rasterio

download_aoi = box(26.78, 38.25, 26.83, 38.29)
pilot_scene = next((s for s in pilot_scenes
    if s.datetime[:10] == '2025-06-10'
    and s.geometry and shape(s.geometry).covers(download_aoi)), None)
if pilot_scene is None:
    raise RuntimeError('10 Haziran için bu küçük alanı kapsayan aday bulunamadı.')

pilot_bands = ['B02', 'B03', 'B04', 'B08']
download_provider = CdseProvider({
    'preprocessing': {'resampling': 'nearest'},
    'download': {'max_auto_size_mb': 10},
})
pilot_workdir = Path(mkdtemp(prefix='sentinel_pilot_'))
pilot_file = download_provider.download(
    pilot_scene, download_aoi, pilot_bands, pilot_workdir, resolution_m=10
)
epsg, expected_bounds, width, height = download_provider._utm_bbox(download_aoi, 10)

with rasterio.open(pilot_file) as src:
    assert src.descriptions == tuple(pilot_bands + ['SCL']), 'Bant sırası beklenenden farklı.'
    assert src.dtypes == ('uint16',) * 5, 'Veri türü beklenenden farklı.'
    assert src.crs and src.crs.to_epsg() == epsg, 'Koordinat sistemi beklenenden farklı.'
    assert (src.width, src.height) == (width, height), 'Görüntü boyutu beklenenden farklı.'
    assert np.allclose(src.bounds, expected_bounds) and np.allclose(src.res, (10, 10)), 'Görüntü gridi beklenenden farklı.'
    assert np.any(src.read([1, 2, 3, 4]) > 0), 'Optik bantlarda veri bulunamadı.'
    scl = src.read(5)
    assert np.isin(scl, np.arange(12)).all() and np.any(scl != 0), 'SCL kalite katmanı geçersiz veya boş.'
    pilot_valid_pct = float(100 * np.mean(~np.isin(scl, [0, 1, 3, 8, 9, 10, 11])))

print('Deneme dosyası Colab içinde doğrulandı.')
print('Dosya boyutu (MB):', round(pilot_file.stat().st_size / 1e6, 2))
print('SCL maskesine göre kullanılabilir piksel (%):', round(pilot_valid_pct, 1))

# 7. Doğrulanan küçük dosyayı Drive’a kaydet

Bu adım görüntüyü yeniden indirmez. Colab’deki `pilot_file` dosyasını projenin `raw/pilot/` klasörüne birebir kopyalar; raster yapısını ve SHA256 özetini Drive kopyasından yeniden okuyarak doğrular. Kaynak dosya korunur.

JSON kaydı; tarih, istenen alan, bantlar, grid, paket sürümü, hash ve katalog adayını içerir. Process API’nin tam UTC gününü filtrelediği, kesin kaynak ürünlerin doğrulanmadığı ve bilimsel kalite kabulünün yapılmadığı kaydedilir. Gönderilmeyen harmonizasyon ayarı, belgelenen API varsayılanından ayrı tutulur.

Eşleşen mevcut dosya ve kayıt yeniden kullanılır. Farklı bir hedef veya kayıt üzerine yazılmaz. Dosya mevcut fakat kayıt eksikse, Colab kaynağıyla birebir eşleşme doğrulanarak kayıt tamamlanabilir. Drive dosya sisteminde yeniden adlandırmaya tam atomiklik garantisi verilmez.

TIFF uyarıları bu adımda gizlenmez veya yeniden yazımla giderilmez. Ham dosyanın baytları korunur; görsel inceleme sonraki adımdır.


In [ ]:
# Run in the existing Colab notebook after the small download:
# %run -i colab_pilot_save.py

import hashlib
import json
import shutil
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import numpy as np
import rasterio
from shapely.geometry import mapping
from sentinel_crop_pipeline.discovery.providers.cdse import CdseProvider


def save_pilot_to_drive(source, project_root, scene, aoi, bands):
    source, project_root = Path(source), Path(project_root)
    if not source.is_file():
        raise FileNotFoundError('Colab geçici dosyası bulunamadı. Drive kaydı tamamlanmadan çalışma zamanı kapanmış olabilir.')
    if not project_root.is_dir():
        raise FileNotFoundError('Drive proje klasörü bulunamadı. Önce Drive bağlantısını kontrol edin.')

    epsg, bounds, width, height = CdseProvider()._utm_bbox(aoi, 10)
    expected_bands = tuple(bands) + ('SCL',)

    def sha256(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    def inspect(path):
        with rasterio.open(path) as src:
            if src.descriptions != expected_bands or src.dtypes != ('uint16',) * len(expected_bands):
                raise ValueError('Raster bantları veya veri türü beklenenden farklı.')
            if not src.crs or src.crs.to_epsg() != epsg or (src.width, src.height) != (width, height):
                raise ValueError('Raster koordinat sistemi veya boyutları beklenenden farklı.')
            if not np.allclose(src.bounds, bounds, rtol=0, atol=1e-6) or not np.allclose(src.res, (10, 10), rtol=0, atol=1e-9):
                raise ValueError('Raster gridi beklenenden farklı.')
            if not np.any(src.read(list(range(1, len(bands) + 1))) > 0):
                raise ValueError('Optik bantlarda veri bulunamadı.')
            scl = src.read(len(expected_bands))
            if not np.isin(scl, np.arange(12)).all() or not np.any(scl != 0):
                raise ValueError('SCL kalite katmanı geçersiz veya boş.')
            return {
                'crs': src.crs.to_string(),
                'width': src.width,
                'height': src.height,
                'bounds': list(src.bounds),
                'transform': list(src.transform)[:6],
                'resolution': list(src.res),
                'band_names': list(src.descriptions),
                'dtypes': list(src.dtypes),
                'scl_usable_pixel_pct': float(100 * np.mean(~np.isin(scl, [0, 1, 3, 8, 9, 10, 11]))),
            }

    raster_info = inspect(source)
    digest = sha256(source)
    day = scene.datetime[:10]
    request = {
        'collection': 'sentinel-2-l2a',
        'endpoint': 'https://sh.dataspace.copernicus.eu/api/v1/process',
        'time_range_utc': {'from': f'{day}T00:00:00Z', 'to': f'{day}T23:59:59Z'},
        'aoi_epsg4326': mapping(aoi),
        'output_resolution_m': 10,
        'bands': list(expected_bands),
        'units': 'DN',
        'sample_type': 'UINT16',
        'resampling': 'NEAREST',
        'harmonizeValues': {'sent_in_request': False, 'documented_api_default': True},
        'mosaickingOrder': {'sent_in_request': False, 'documented_api_default': 'mostRecent'},
    }
    request_id = hashlib.sha256(json.dumps(request, sort_keys=True).encode('utf-8')).hexdigest()[:16]
    output_dir = project_root / 'raw' / 'pilot'
    output_dir.mkdir(parents=True, exist_ok=True)
    target = output_dir / f'urla_pilot_{day.replace("-", "")}_{request_id}.tif'
    record = target.with_suffix('.json')

    if record.exists():
        try:
            existing = json.loads(record.read_text(encoding='utf-8'))
        except (ValueError, OSError):
            raise RuntimeError('Mevcut kontrol kaydı okunamadı; üzerine yazılmadı.') from None
        if not isinstance(existing, dict) or existing.get('sha256') != digest or existing.get('request_id') != request_id:
            raise RuntimeError('Mevcut kontrol kaydı farklı bir dosyaya ait; üzerine yazılmadı.')

    partial = target.with_name(target.name + f'.{uuid4().hex}.partial')
    partial_record = record.with_name(record.name + f'.{uuid4().hex}.partial')
    try:
        if not target.exists():
            shutil.copyfile(source, partial)
            inspect(partial)
            if sha256(partial) != digest:
                raise RuntimeError('Drive aktarımının SHA256 kontrolü başarısız.')
            if not target.exists():
                partial.rename(target)
        if sha256(target) != digest:
            raise RuntimeError('Drive hedefinde farklı veya eksik bir dosya var; üzerine yazılmadı.')
        drive_info = inspect(target)
        if drive_info != raster_info:
            raise RuntimeError('Drive kopyasının raster bilgileri kaynakla eşleşmiyor.')

        if not record.exists():
            metadata = {
                'schema_version': 1,
                'purpose': 'technical_pilot_not_a_final_training_dataset',
                'created_utc': datetime.now(timezone.utc).isoformat(),
                'package_version': version('sentinel-crop-pipeline'),
                'request_id': request_id,
                'request': request,
                'candidate_reference': {
                    'id': scene.id,
                    'datetime': scene.datetime,
                    'cloud_cover_pct': scene.cloud_cover,
                    'geometry': scene.geometry,
                },
                'exact_source_product_ids_verified': False,
                'source_selection_note': 'Process API filters the full UTC day; the candidate reference is not verified as the exclusive source.',
                'actual_source_processing_baselines': None,
                'processing_documentation': 'https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html',
                'raster': drive_info,
                'sha256': digest,
                'size_bytes': target.stat().st_size,
                'project_relative_path': target.relative_to(project_root).as_posix(),
                'pixels_rewritten_or_normalized': False,
                'scientific_quality_accepted': False,
            }
            partial_record.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
            if not record.exists():
                partial_record.rename(record)
        persisted = json.loads(record.read_text(encoding='utf-8'))
        if persisted.get('sha256') != digest or persisted.get('request_id') != request_id:
            raise RuntimeError('Drive kontrol kaydının doğrulaması başarısız.')
    finally:
        for staging_file in (partial, partial_record):
            if staging_file.is_file():
                staging_file.unlink()

    print('Drive kopyası ve SHA256 kontrolü tamamlandı.')
    print('Görüntü:', target.relative_to(project_root).as_posix())
    print('Kontrol kaydı:', record.relative_to(project_root).as_posix())
    return target


if __name__ == '__main__':
    pilot_drive_file = save_pilot_to_drive(
        pilot_file, PROJECT_ROOT, pilot_scene, download_aoi, pilot_bands
    )


# 8. Drive kopyasını görsel olarak incele

Doğrulanan Drive dosyasından B04/B03/B02 bantları açıkça seçilir. SCL maskesine göre geçerli piksellerin ortak %2 ve %98 yüzdelikleri yalnızca ekran kontrastını ayarlamak için kullanılır. Ham TIFF değiştirilmez; bu germe model girdisi normalizasyonu değildir.

Önizleme PNG’si `results/pilot/` altında saklanır. Görselin düzgün olması ürün türü etiketlerini veya veri kümesinin eğitim için yeterli olduğunu doğrulamaz. Sonraki planlama adımı, etiket kaynağı ve buna uygun çalışma yılını belirlemektir.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio

with rasterio.open(pilot_drive_file) as src:
    ids = [src.descriptions.index(b) + 1 for b in ('B04', 'B03', 'B02')]
    rgb = src.read(ids).astype(np.float32)
    scl = src.read(src.descriptions.index('SCL') + 1)
    valid = ~np.isin(scl, [0, 1, 3, 8, 9, 10, 11])

if not valid.any():
    raise ValueError('Bu görüntüde SCL maskesine göre kullanılabilir piksel yok.')
lo, hi = np.percentile(rgb[:, valid], [2, 98])
if hi <= lo:
    raise ValueError('Önizleme için yeterli renk farkı bulunamadı.')
rgb = np.clip((rgb - lo) / (hi - lo), 0, 1)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(np.moveaxis(rgb, 0, -1), alpha=valid.astype(float), interpolation='nearest')
ax.set_title('Teknik pilot - 10 Haziran 2025 (RGB)')
ax.axis('off')
preview_dir = PROJECT_ROOT / 'results' / 'pilot'
preview_dir.mkdir(parents=True, exist_ok=True)
preview_png = preview_dir / (pilot_drive_file.stem + '_rgb.png')
fig.savefig(preview_png, dpi=150, bbox_inches='tight')
plt.show()
print('Önizleme kaydedildi:', preview_png.relative_to(PROJECT_ROOT))
